In [ ]:
# Ch05: 에이전트 아키텍처 — Function Calling
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower', 'anthropic'])
import pandapower as pp
import pandapower.networks as pn
import json
import numpy as np

try:
    sys.path.insert(0, '..')
    from utils.style import set_style
    set_style()
except:
    pass

## 1. 도구 정의 (JSON Schema)

In [ ]:
tools = [
    {
        "name": "run_power_flow",
        "description": "IEEE 테스트 시스템에서 조류 계산을 실행합니다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "system": {
                    "type": "string",
                    "enum": ["ieee14", "ieee30", "ieee39"],
                    "description": "IEEE 테스트 시스템"
                },
                "load_changes": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "bus": {"type": "integer"},
                            "p_mw": {"type": "number"},
                            "q_mvar": {"type": "number"}
                        },
                        "required": ["bus"]
                    },
                    "description": "변경할 부하 목록"
                }
            },
            "required": ["system"]
        }
    },
    {
        "name": "get_voltage_violations",
        "description": "조류 계산 결과에서 전압 위반 모선을 찾습니다.",
        "input_schema": {
            "type": "object",
            "properties": {
                "v_min": {"type": "number", "default": 0.95},
                "v_max": {"type": "number", "default": 1.05}
            }
        }
    }
]
print(f"정의된 도구: {[t['name'] for t in tools]}")

## 2. 도구 실행 함수

In [ ]:
net = None

def execute_tool(name, inputs):
    global net
    if name == "run_power_flow":
        loader = {"ieee14": pn.case14, "ieee30": pn.case_ieee30, "ieee39": pn.case39}
        net = loader[inputs["system"]]()
        for change in inputs.get("load_changes", []):
            bus = change["bus"]
            mask = net.load["bus"] == bus
            if mask.any():
                if "p_mw" in change:
                    net.load.loc[mask, "p_mw"] = change["p_mw"]
                if "q_mvar" in change:
                    net.load.loc[mask, "q_mvar"] = change["q_mvar"]
        pp.runpp(net)
        return json.dumps({
            "converged": bool(net.converged),
            "voltages": {str(k): round(v, 4) for k, v in net.res_bus["vm_pu"].to_dict().items()},
            "line_loadings": {str(k): round(v, 2) for k, v in net.res_line["loading_percent"].to_dict().items()}
        })
    elif name == "get_voltage_violations":
        v_min = inputs.get("v_min", 0.95)
        v_max = inputs.get("v_max", 1.05)
        violations = net.res_bus[
            (net.res_bus["vm_pu"] < v_min) | (net.res_bus["vm_pu"] > v_max)
        ]["vm_pu"].round(4).to_dict()
        return json.dumps({
            "violations": {str(k): v for k, v in violations.items()},
            "total": len(violations),
            "min_voltage": round(float(net.res_bus["vm_pu"].min()), 4),
            "max_voltage": round(float(net.res_bus["vm_pu"].max()), 4)
        })

## 3. 도구 직접 테스트

In [ ]:
# 도구를 직접 호출하여 테스트
result1 = execute_tool("run_power_flow", {"system": "ieee14"})
print("=== 기본 조류 계산 ===")
r1 = json.loads(result1)
print(f"수렴: {r1['converged']}")
print(f"전압 범위: {min(r1['voltages'].values()):.4f} ~ {max(r1['voltages'].values()):.4f} p.u.")

# 부하 변경 후
result2 = execute_tool("run_power_flow", {
    "system": "ieee14",
    "load_changes": [{"bus": 4, "p_mw": 15.0}]
})
r2 = json.loads(result2)
print(f"\n=== 부하 변경 후 ===")
print(f"수렴: {r2['converged']}")
print(f"전압 범위: {min(r2['voltages'].values()):.4f} ~ {max(r2['voltages'].values()):.4f} p.u.")

# 전압 위반 확인
result3 = execute_tool("get_voltage_violations", {"v_min": 0.95, "v_max": 1.05})
r3 = json.loads(result3)
print(f"\n=== 전압 위반 ===")
print(f"위반 모선 수: {r3['total']}")
print(f"최소 전압: {r3['min_voltage']} p.u.")

## 4. ReAct 루프 (시뮬레이션)

In [ ]:
# API 없이 에이전트 동작을 시뮬레이션
print("=== 에이전트 실행 시뮬레이션 ===\n")

question = "IEEE 14-bus에서 5번 모선 부하를 50% 증가시키면 전압 위반이 발생하나요?"
print(f"[User] {question}\n")

# Turn 1: 기본 시스템 로드
print("[LLM] 먼저 기본 시스템을 로드하겠습니다.")
print("      → run_power_flow(system='ieee14')")
result = execute_tool("run_power_flow", {"system": "ieee14"})
r = json.loads(result)
bus4_base_load = net.load.loc[net.load['bus'] == 4, 'p_mw'].values[0]
print(f"[Tool] 수렴: {r['converged']}, 5번 모선 기본 부하: {bus4_base_load:.1f} MW\n")

# Turn 2: 부하 50% 증가
new_load = bus4_base_load * 1.5
print(f"[LLM] 5번 모선 부하를 50% 증가시킵니다 ({bus4_base_load:.1f} → {new_load:.1f} MW)")
print(f"      → run_power_flow(system='ieee14', load_changes=[{{bus:4, p_mw:{new_load:.1f}}}])")
result = execute_tool("run_power_flow", {
    "system": "ieee14",
    "load_changes": [{"bus": 4, "p_mw": new_load}]
})
r = json.loads(result)
print(f"[Tool] 수렴: {r['converged']}\n")

# Turn 3: 전압 위반 확인
print("[LLM] 전압 위반 여부를 확인하겠습니다.")
print("      → get_voltage_violations(v_min=0.95, v_max=1.05)")
result = execute_tool("get_voltage_violations", {"v_min": 0.95, "v_max": 1.05})
r = json.loads(result)
print(f"[Tool] 위반: {r['total']}건, 최소전압: {r['min_voltage']} p.u.\n")

# Final answer
if r['total'] == 0:
    print(f"[LLM] 5번 모선 부하를 50% 증가시켜도 전압 위반은 발생하지 않습니다.")
    print(f"      최저 전압은 {r['min_voltage']} p.u.로 0.95 기준을 만족합니다.")
else:
    print(f"[LLM] 전압 위반이 {r['total']}건 발생했습니다: {r['violations']}")

## 5. Claude Code 방식: pandapower 직접 실행

In [ ]:
# Demo 5.2: Claude Code처럼 직접 실행
net = pn.case14()
pp.runpp(net)

print("=== IEEE 14-Bus 조류 계산 결과 ===")
print(f"수렴 여부: {net.converged}")
print(f"\n--- 모선 전압 [p.u.] ---")
print(net.res_bus[["vm_pu", "va_degree"]].round(4))
print(f"\n총 발전: {net.res_ext_grid['p_mw'].sum() + net.res_gen['p_mw'].sum():.1f} MW")
print(f"총 부하: {net.load['p_mw'].sum():.1f} MW")
print(f"총 손실: {net.res_line['pl_mw'].sum():.2f} MW")